# Train and validate the sex classifier

Research only: fine-tune an ImageNet-pretrained ResNet-18 on human-labelled crops
and validate it by holding out one recording date at a time. Human labels provide
the training targets and evaluation reference; each prediction uses its own crop.
No expected male count is supplied to the model.

Run from `notebooks/` with `../data/labeled_crops.npz`, or upload that file into the
working directory on Colab. Use only trusted dataset files: crop provenance is
stored as a NumPy object array and requires `allow_pickle=True`.

The later cells train six validation models and two deployment models with the
default settings. A GPU is recommended. The first model construction downloads
ImageNet weights unless they are already cached. Routine scoring uses saved model
weights and does not require this notebook. Deployment weights are saved in
`../model/`; copy them deliberately if replacing a released model.


## Imports and configuration

In [ ]:
# Import dependencies
import collections
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torchvision import models


In [ ]:
# Find a local dataset or a Colab upload
def find_data_file(name):
    """Find a local dataset or a Colab upload."""
    # Process each candidate
    for candidate in (Path("../data") / name, Path(name)):
        if candidate.exists():
            # Return the candidate
            return candidate
    raise ValueError(f"Missing {name}; put it in ../data or the current directory")


# Define file locations
dataset_path = find_data_file(name="labeled_crops.npz")
model_directory = Path("../model")

# Define training hyperparameters
epochs = 40
batch_size = 64
learning_rate = 6e-4
weight_decay = 1e-2
frozen_layers = 6
seeds = (0, 1)
test_time_augmentations = 6

# Call a fly male above this score, and call the score confident outside the band
decision_threshold = 0.5
confident_low = 0.1
confident_high = 0.9

# Define augmentation strength
rotation_limit = 8 * np.pi / 180
translation_limit = 0.12
scale_range = (0.92, 1.08)
brightness_range = (0.85, 1.15)
brightness_shift = 0.14


In [ ]:
# Select the compute device
def select_device():
    """Select the compute device."""
    # Prefer CUDA, then Apple Silicon, then CPU
    if torch.cuda.is_available():
        # Return the result
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        # Return the result
        return torch.device("mps")
    # Return the result
    return torch.device("cpu")


# Prepare the device
device = select_device()
device_name = (
    torch.cuda.get_device_name(device=0) if device.type == "cuda" else device.type
)
print(f"Device: {device_name}")


## Function definitions

In [ ]:
# Build a single-channel ResNet-18 from ImageNet weights
def build_model(frozen_layers):
    """Build a single-channel ResNet-18 from ImageNet weights."""
    # Load the pretrained network
    network = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Collapse the three-channel stem into one channel
    summed_weights = network.conv1.weight.data.sum(
        dim=1,
        keepdim=True,
    )
    network.conv1 = nn.Conv2d(
        in_channels=1,
        out_channels=64,
        kernel_size=7,
        stride=2,
        padding=3,
        bias=False,
    )
    network.conv1.weight.data = summed_weights

    # Freeze the early layers
    for layer in list(network.children())[:frozen_layers]:
        for parameter in layer.parameters():
            parameter.requires_grad = False

    # Replace the classifier head
    network.fc = nn.Linear(
        in_features=512,
        out_features=1,
    )

    # Return the adapted network
    return network


In [ ]:
# Apply random augmentations to a batch
def augment_batch(images):
    """Apply random augmentations to a batch."""
    # Flip the batch vertically
    if torch.rand(1) < 0.5:
        images = torch.flip(
            input=images,
            dims=[2],
        )

    # Flip the batch horizontally
    if torch.rand(1) < 0.5:
        images = torch.flip(
            input=images,
            dims=[3],
        )

    # Sample per-image affine parameters
    batch_size = len(images)
    angles = (
        (
            torch.rand(
                batch_size,
                device=images.device,
            )
            - 0.5
        )
        * 2
        * rotation_limit
    )
    scales = scale_range[0] + (scale_range[1] - scale_range[0]) * torch.rand(
        batch_size,
        device=images.device,
    )
    shift_x = (
        torch.rand(
            batch_size,
            device=images.device,
        )
        - 0.5
    ) * translation_limit
    shift_y = (
        torch.rand(
            batch_size,
            device=images.device,
        )
        - 0.5
    ) * translation_limit

    # Assemble the affine matrices
    cosine = torch.cos(angles) / scales
    sine = torch.sin(angles) / scales
    affine_matrices = torch.stack(
        tensors=[
            torch.stack(
                tensors=[cosine, -sine, shift_x],
                dim=1,
            ),
            torch.stack(
                tensors=[sine, cosine, shift_y],
                dim=1,
            ),
        ],
        dim=1,
    )

    # Resample the batch
    grid = nn.functional.affine_grid(
        theta=affine_matrices,
        size=images.shape,
        align_corners=False,
    )
    images = nn.functional.grid_sample(
        input=images,
        grid=grid,
        align_corners=False,
        padding_mode="border",
    )

    # Jitter brightness
    gain = brightness_range[0] + (
        brightness_range[1] - brightness_range[0]
    ) * torch.rand(
        batch_size,
        1,
        1,
        1,
        device=images.device,
    )
    offset = brightness_shift * (
        torch.rand(
            batch_size,
            1,
            1,
            1,
            device=images.device,
        )
        - 0.5
    )

    # Return the augmented batch
    return images * gain + offset


In [ ]:
# Train one model on the given crops
def train_model(train_images, train_labels, seed):
    """Train one model on the given crops."""
    # Seed the run
    torch.manual_seed(seed=seed)

    # Build the network and optimiser
    network = build_model(frozen_layers=frozen_layers).to(device)
    trainable = [
        parameter for parameter in network.parameters() if parameter.requires_grad
    ]
    optimizer = torch.optim.AdamW(
        params=trainable,
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    # Weight the loss by the class imbalance
    positive_weight = torch.tensor(
        data=[float((train_labels == 0).sum()) / float((train_labels == 1).sum())],
        dtype=torch.float32,
        device=device,
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)

    # Move the data onto the device
    image_tensor = torch.tensor(
        data=train_images,
        device=device,
    ).unsqueeze(dim=1)
    label_tensor = torch.tensor(
        data=train_labels,
        device=device,
    )

    # Schedule the learning rate over the whole run
    steps_per_epoch = max(
        1,
        len(image_tensor) // batch_size,
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer=optimizer,
        max_lr=learning_rate,
        total_steps=epochs * steps_per_epoch,
    )

    # Run the training epochs
    for epoch in range(epochs):
        network.train()
        order = torch.randperm(
            len(image_tensor),
            device=device,
        )
        for start in range(
            0,
            len(order) - batch_size + 1,
            batch_size,
        ):
            batch_indices = order[start : start + batch_size]
            optimizer.zero_grad(set_to_none=True)
            logits = network(
                x=augment_batch(images=image_tensor[batch_indices])
            ).squeeze(1)
            loss = criterion(
                input=logits,
                target=label_tensor[batch_indices],
            )
            loss.backward()
            optimizer.step()
            scheduler.step()

    # Return the trained network
    return network


In [ ]:
# Average one network's predictions over several augmentations
def predict_with_augmentation(network, test_images):
    """Average one network's predictions over several augmentations."""
    # Move the crops onto the device
    test_tensor = torch.tensor(
        data=test_images,
        device=device,
    ).unsqueeze(dim=1)

    # Score the clean crops, then several augmented copies
    network.eval()
    predictions = []
    with torch.no_grad():
        clean = torch.sigmoid(input=network(x=test_tensor).squeeze(1))
        predictions.append(clean.cpu().numpy())

        # Process each
        for _ in range(test_time_augmentations - 1):
            scores = torch.sigmoid(
                input=network(x=augment_batch(images=test_tensor)).squeeze(1)
            )
            predictions.append(scores.cpu().numpy())

    # Return the mean probability per fly
    return np.mean(
        a=predictions,
        axis=0,
    )


In [ ]:
# Average predictions over several seeds
def fit_ensemble(train_images, train_labels, test_images, seeds):
    """Average predictions over several seeds."""
    # Train one model per seed and score the held-out crops with each
    predictions = []
    for seed in seeds:
        network = train_model(
            train_images=train_images,
            train_labels=train_labels,
            seed=seed,
        )
        predictions.append(
            predict_with_augmentation(
                network=network,
                test_images=test_images,
            )
        )

    # Return the mean probability
    return np.mean(
        a=predictions,
        axis=0,
    )


## Load the dataset

In [ ]:
# Load the crops and their provenance
dataset = np.load(
    file=dataset_path,
    allow_pickle=True,
)
crops = dataset["crops"]
labels = dataset["y"].astype(np.float32)
sources = dataset["meta"]

# Validate crop, label, and provenance alignment
if crops.ndim != 3 or crops.shape[1:] != (96, 96):
    raise ValueError(f"Expected crops with shape (n, 96, 96), got {crops.shape}")
if labels.shape != (len(crops),) or sources.shape != (len(crops), 4):
    raise ValueError("Crop, label, and provenance lengths must match")
if not np.isin(
    element=labels,
    test_elements=[0, 1],
).all():
    raise ValueError("Training labels must contain only 0 and 1")
if (
    min(
        int((labels == 0).sum()),
        int((labels == 1).sum()),
    )
    < 8
):
    raise ValueError(
        "At least eight labelled crops per class are required for this notebook"
    )

# Derive the grouping columns
wells = np.asarray(
    a=[row[0] for row in sources],
    dtype=str,
)
frames = np.asarray(
    a=[row[2] for row in sources],
    dtype=np.int64,
)
dates = np.asarray(
    a=[well[:4] for well in wells],
    dtype=str,
)

# Normalise the images for the network
images = (crops.astype(np.float32) / 255.0 - 0.5) / 0.25

# Report the results
print(
    f"Dataset: {len(labels)} crops | {int(labels.sum())} male | {int((1 - labels).sum())} female"
)
print(f"")
for date in sorted(set(dates)):
    selected = dates == date
    well_list = ", ".join(sorted(set(wells[selected])))
    print(f"  {date} | {selected.sum():4d} crops | {well_list}")


## Sanity check the crops

In [ ]:
# Show a sample of each class
figure, axes = plt.subplots(
    nrows=2,
    ncols=8,
    figsize=(14, 3.8),
)
generator = np.random.default_rng(seed=0)

# Process each row, (label value, label name
for row, (label_value, label_name) in enumerate([(1, "male"), (0, "female")]):
    sample = generator.choice(
        np.where(labels == label_value)[0],
        8,
        replace=False,
    )
    for column, index in enumerate(sample):
        axes[row, column].imshow(
            X=crops[index],
            cmap="gray",
        )
        axes[row, column].axis("off")
    axes[row, 0].set_title(
        label=label_name,
        loc="left",
        fontsize=11,
    )

# Arrange and display the training examples
plt.tight_layout()
plt.show()


## Validate by holding out one recording date

Consecutive frames of the same fly are near-duplicates, so validation holds out
whole recording dates. This estimates performance across the dates represented
in the dataset; it does not establish accuracy on every future recording.

A binary evaluation call uses a score above 0.5. The confident share is the
fraction of crops outside the 0.1 to 0.9 score band. Confidence is a model score
heuristic; inspect both its coverage and held-out accuracy before choosing a
threshold. Routine inference exposes each fly's score and rank without a count quota.


In [ ]:
# Hold out each date in turn
fold_results = []
held_out_probabilities = np.full(
    shape=len(labels),
    fill_value=np.nan,
    dtype=np.float32,
)
held_out_dates = sorted(set(dates))

# Process each held out
for held_out in held_out_dates:
    # Split by recording date
    test_selector = dates == held_out
    train_selector = ~test_selector

    # Train an ensemble and predict the held-out date
    probabilities = fit_ensemble(
        train_images=images[train_selector],
        train_labels=labels[train_selector],
        test_images=images[test_selector],
        seeds=seeds,
    )
    held_out_probabilities[test_selector] = probabilities

    # Call every fly on its own score
    calls = (probabilities > decision_threshold).astype(int)
    truth = labels[test_selector]

    # Score the fold, and score it again on the confident calls alone
    confident = (probabilities < confident_low) | (probabilities > confident_high)
    fold_results.append(
        {
            "held_out": held_out,
            "crops": int(test_selector.sum()),
            "auc": roc_auc_score(
                y_true=truth,
                y_score=probabilities,
            ),
            "accuracy": float((calls == truth).mean()),
            "confident_share": float(confident.mean()),
            "confident_accuracy": float((calls[confident] == truth[confident]).mean()),
        }
    )

    # Report the results
    print(
        f"Fold {len(fold_results)} | "
        f"Held out 2026-{held_out[:2]}-{held_out[2:]} | "
        f"AUC: {fold_results[-1]['auc']:.3f} | "
        f"Accuracy: {fold_results[-1]['accuracy']:.3f} | "
        f"Confident {fold_results[-1]['confident_share']:.0%} of crops at {fold_results[-1]['confident_accuracy']:.3f}"
    )


In [ ]:
# Report the mean across folds
for name in ("auc", "accuracy", "confident_share", "confident_accuracy"):
    print(
        f"Mean {name.replace('_', ' ',)}: {np.mean(a=[result[name] for result in fold_results]):.3f}"
    )

# Save the held-out predictions next to the dataset
np.savez_compressed(
    file=dataset_path.with_name("cv_predictions.npz"),
    meta=sources,
    labels=labels,
    probabilities=held_out_probabilities,
    calls=(held_out_probabilities > decision_threshold).astype(int),
    dates=dates,
)
print(f"Saved {dataset_path.with_name('cv_predictions.npz')}")


## Train the deployment model on every labelled fly

In [ ]:
# Train and save one model per seed
model_directory.mkdir(
    parents=True,
    exist_ok=True,
)

# Process each seed
for seed in seeds:
    network = train_model(
        train_images=images,
        train_labels=labels,
        seed=seed,
    )
    checkpoint_path = model_directory / f"flysex_seed{seed}.pt"
    torch.save(
        obj=network.state_dict(),
        f=checkpoint_path,
    )
    print(f"Saved {checkpoint_path}")
